# Actividad 2 · Big Data
### Titanic · Retrasos aéreos · Viajes de taxi en Nueva York

**Institución Universitaria de Envigado — Big Data**
**Docente:** Andrés Felipe Hernández Marulanda
**Integrantes:** Natalia Flores Pérez · Santiago Arcila Gutiérrez · Alejandro Restrepo Uribe

---

### Contenido

| Parte | Conjunto de datos | Tamaño | Qué se pide |
|---|---|---|---|
| 1 | Titanic | 891 filas | Clasificar supervivencia por género, clase social y edad |
| 2 | Flight Delays (2015) | 5.819.079 filas | Top 10 aerolíneas con más retrasos y variación mensual |
| 3 | NYC Taxi Trip Duration | 1.458.644 filas | Qué es Dask, distribución de tarifas y viajes por hora |

Las partes 2 y 3 se resuelven con **Dask** porque los archivos no caben cómodamente
en memoria. La parte 1 se hace con pandas y scikit-learn, que es lo apropiado para
891 filas: usar Dask ahí sería más lento, no más rápido.

---

### Nota previa sobre el enunciado

Durante el desarrollo encontramos dos puntos del enunciado que requieren una
interpretación explícita. Las dejamos documentadas aquí en lugar de resolverlas en
silencio, porque afectan lo que se entrega.

**1. "Clasificar sobrevivientes hombres vs sobrevivientes mujeres".** Leído de forma
literal, separar a los sobrevivientes por género no requiere un algoritmo: es un filtro
sobre una columna. Entendemos que lo que se pide es **entrenar un clasificador de
supervivencia y analizar el papel del género**, que es el ejercicio con contenido
estadístico. Lo mismo aplica a las tareas de clase social y edad. Bajo esa lectura,
las tres tareas del Titanic se resuelven como tres modelos de clasificación, cada uno
centrado en una variable, más un modelo completo que las combina.

**2. El conjunto de taxis enlazado no contiene tarifas.** El enunciado pide analizar
"la distribución de las tarifas de taxi" y enlaza la competencia
`nyc-taxi-trip-duration`. Ese conjunto trae identificador, proveedor, fechas de
recogida y entrega, número de pasajeros, coordenadas y **duración del viaje**; no tiene
ninguna columna de tarifa. El conjunto con tarifas es otro
(*New York City Taxi Fare Prediction*).

Resolvimos así: analizamos la distribución de la **duración de los viajes**, que es la
variable de costo disponible en el conjunto solicitado, y además **estimamos la tarifa**
aplicando la fórmula oficial de la Taxi & Limousine Commission de Nueva York para 2016.
Así se responde la pregunta tal como fue formulada, sin cambiar de conjunto de datos y
dejando claro qué es dato medido y qué es estimación nuestra.

## Parte 0 · Preparación del entorno

In [ ]:
# Dask no viene preinstalado en Google Colab con el submódulo de DataFrame.
# Si esta celda falla, ejecutar y reiniciar el entorno de ejecución.
try:
    import dask.dataframe as dd
except ImportError:
    !pip install -q "dask[dataframe]"
    import dask.dataframe as dd

print("Dask disponible")

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

SEMILLA = 42
np.random.seed(SEMILLA)

# Rutas. Ajustar si los archivos están en otra carpeta.
RUTA_TITANIC = 'data/titanic.csv'
RUTA_VUELOS  = 'data/flights.csv'
RUTA_TAXIS   = 'data/taxi_train.csv'

for nombre, ruta in [('Titanic', RUTA_TITANIC), ('Vuelos', RUTA_VUELOS), ('Taxis', RUTA_TAXIS)]:
    existe = os.path.exists(ruta)
    tam = f"{os.path.getsize(ruta)/1e6:,.0f} MB" if existe else '—'
    print(f"{nombre:<9} {'OK ' if existe else 'FALTA'}  {ruta:<22} {tam}")

---
---
# Parte 1 · Titanic

Tres algoritmos de clasificación de supervivencia, cada uno centrado en una variable
distinta (género, clase social, edad), más un modelo completo que las combina para
medir cuánto aporta cada una.

## 1.1 Exploración del conjunto

In [ ]:
titanic = pd.read_csv(RUTA_TITANIC)
print(f"Filas: {titanic.shape[0]}   Columnas: {titanic.shape[1]}")
titanic.head()

In [ ]:
resumen = pd.DataFrame({
    'tipo':    [str(t) for t in titanic.dtypes],
    'nulos':   titanic.isna().sum().values,
    '% nulos': (titanic.isna().mean()*100).round(1).values,
    'únicos':  [titanic[c].nunique() for c in titanic.columns],
    'ejemplo': [titanic[c].dropna().iloc[0] for c in titanic.columns],
})
resumen

### Observaciones sobre este archivo

Este **no es el CSV estándar de Kaggle**. El de Kaggle trae `PassengerId`, `SibSp`,
`Parch`, `Ticket` y `Cabin`; este trae apellido y nombre por separado, y la
supervivencia como texto `yes`/`no` en lugar de 1/0. Conviene decirlo en la
sustentación, porque cualquier tutorial de internet asume el otro formato.

Dos cosas que condicionan el trabajo:

- **`age` tiene 177 nulos, casi el 20%.** Eso pega directo en la tercera tarea, que es
  justamente la de edad. No se puede ignorar ni borrar sin más: eliminar esas filas
  costaría una de cada cinco observaciones.
- **La columna `first` contiene el tratamiento** (`Mr.`, `Mrs.`, `Miss.`, `Master.`).
  Eso es información aprovechable: `Master.` identifica niños varones y `Miss.` a
  mujeres jóvenes o solteras, lo que permite imputar edades mucho mejor que con una
  mediana global.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

sup = titanic['survived'].value_counts()
ax[0].bar(['No sobrevivió','Sobrevivió'], [sup['no'], sup['yes']],
          color=['#C44E52','#55A868'])
ax[0].set_title(f"Supervivencia general ({sup['yes']/len(titanic):.1%} sobrevivió)")
ax[0].set_ylabel('Nº de pasajeros')

ax[1].hist(titanic['age'].dropna(), bins=35, color='#4C72B0', edgecolor='white')
ax[1].set_title(f"Edad ({titanic['age'].isna().sum()} nulos de {len(titanic)})")
ax[1].set_xlabel('Años')

ax[2].hist(titanic['fare'], bins=45, color='#DD8452', edgecolor='white')
ax[2].set_title('Tarifa pagada')
ax[2].set_xlabel('Libras')

plt.tight_layout(); plt.show()

In [ ]:
# Tasas de supervivencia por cada variable candidata
titanic['sobrevivio'] = (titanic['survived'] == 'yes').astype(int)

for col, etiqueta in [('gender','Género'), ('class','Clase'), ('embarked','Puerto')]:
    t = titanic.groupby(col).agg(pasajeros=('sobrevivio','size'),
                                 sobrevivieron=('sobrevivio','sum'))
    t['tasa'] = (t['sobrevivieron']/t['pasajeros']*100).round(1)
    print(f"--- {etiqueta} ---")
    print(t.to_string(), "\n")

Los números ya anticipan el resultado de las tres tareas: **74,2% de las mujeres
sobrevivió frente al 18,9% de los hombres**, y la tasa cae de 63% en primera clase a
24% en tercera. La regla "mujeres y niños primero" no es un mito literario, está en los
datos.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

sns.barplot(data=titanic, x='gender', y='sobrevivio', ax=ax[0], errorbar=None,
            palette=['#DD8452','#4C72B0'])
ax[0].set(title='Supervivencia por género', ylabel='Tasa de supervivencia', ylim=(0,1))

sns.barplot(data=titanic, x='class', y='sobrevivio', ax=ax[1], errorbar=None,
            palette='Blues_r')
ax[1].set(title='Supervivencia por clase social', ylabel='', ylim=(0,1))

sns.barplot(data=titanic, x='gender', y='sobrevivio', hue='class', ax=ax[2],
            errorbar=None, palette='Blues_r')
ax[2].set(title='Género y clase combinados', ylabel='', ylim=(0,1))
ax[2].legend(title='Clase')

plt.tight_layout(); plt.show()

El tercer panel es el más informativo: **una mujer de tercera clase tuvo mejor
pronóstico que un hombre de primera**. El género pesó más que el dinero. Ese contraste
es un buen punto para la sustentación.

## 1.2 Ingeniería de características e imputación de la edad

In [ ]:
# El tratamiento (Mr., Mrs., Miss., Master.) está dentro de la columna 'first'
import re

def extraer_titulo(nombre):
    m = re.search(r'\b(Mr|Mrs|Miss|Master|Dr|Rev|Col|Major|Mlle|Mme|Ms|Capt|Don|Dona|Jonkheer|Lady|Sir|Countess)\.?',
                  str(nombre))
    return m.group(1) if m else 'Otro'

titanic['titulo'] = titanic['first'].apply(extraer_titulo)

# Agrupamos los títulos raros: con 1 o 2 casos no aportan y solo generan ruido
frecuentes = ['Mr','Mrs','Miss','Master']
titanic['titulo'] = titanic['titulo'].where(titanic['titulo'].isin(frecuentes), 'Otro')

print(titanic.groupby('titulo').agg(
    pasajeros=('sobrevivio','size'),
    edad_mediana=('age','median'),
    tasa_superv=('sobrevivio','mean')).round(2).to_string())

Ahí está el valor del título: la edad mediana de `Master.` es muy baja porque designa a
**niños varones**, y su tasa de supervivencia es mucho más alta que la de `Mr.`. Es la
evidencia de "los niños primero", y además nos da una forma sensata de imputar edades.

**Decisión de imputación:** rellenamos la edad faltante con **la mediana de su propio
título**, no con la mediana global. Imputar con la mediana global le pondría unos 28
años a un niño cuyo título dice `Master.`, lo que introduciría un error sistemático
justo en la variable que la tarea 3 quiere estudiar.

In [ ]:
antes_nulos = titanic['age'].isna().sum()
mediana_global = titanic['age'].median()

titanic['edad'] = titanic.groupby('titulo')['age'].transform(lambda s: s.fillna(s.median()))
titanic['edad'] = titanic['edad'].fillna(mediana_global)   # por si algún grupo quedara vacío
titanic['edad_imputada'] = titanic['age'].isna().astype(int)  # bandera de trazabilidad

print(f"Edades imputadas: {antes_nulos} ({antes_nulos/len(titanic):.1%})")
print(f"Mediana global (NO usada como relleno único): {mediana_global:.1f} años")
print("\nMediana usada por título:")
print(titanic.groupby('titulo')['age'].median().round(1).to_string())

In [ ]:
# Comparación: qué habría pasado con la imputación ingenua
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

ax[0].hist(titanic['age'].dropna(), bins=35, color='#4C72B0', edgecolor='white')
ax[0].set_title(f'Edad original (sin los {antes_nulos} nulos)')
ax[0].set_xlabel('Años')

ax[1].hist(titanic['edad'], bins=35, color='#55A868', edgecolor='white')
ax[1].axvline(mediana_global, color='crimson', ls='--',
              label=f'mediana global = {mediana_global:.0f}')
ax[1].set_title('Edad tras imputar por título')
ax[1].set_xlabel('Años'); ax[1].legend()

plt.tight_layout(); plt.show()

print("Si hubiéramos imputado todo con la mediana global, habría aparecido una barra")
print(f"artificial de {antes_nulos} personas justo en {mediana_global:.0f} años.")

In [ ]:
# Grupos de edad, para el análisis y para el modelo de la tarea 3
titanic['grupo_edad'] = pd.cut(
    titanic['edad'],
    bins=[0, 12, 18, 30, 45, 60, 100],
    labels=['Niño (0-12)','Adolesc. (13-18)','Joven (19-30)',
            'Adulto (31-45)','Maduro (46-60)','Mayor (60+)'])

t = titanic.groupby('grupo_edad', observed=True).agg(
    pasajeros=('sobrevivio','size'), tasa=('sobrevivio','mean'))
t['tasa'] = (t['tasa']*100).round(1)
print(t.to_string())

plt.figure(figsize=(9,4.5))
sns.barplot(data=titanic, x='grupo_edad', y='sobrevivio', errorbar=None, palette='viridis')
plt.title('Supervivencia por grupo de edad'); plt.ylabel('Tasa de supervivencia')
plt.xlabel(''); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

La edad **no** tiene una relación lineal con la supervivencia: los niños se salvan
mucho más, después la tasa baja y se mantiene bastante plana entre los 19 y los 60
años. Eso importa para elegir modelo: una regresión logística, que asume una relación
monótona, va a captar mal este patrón; un árbol, que parte por umbrales, lo capta bien.

## 1.3 Los tres algoritmos que pide el enunciado

Definimos una función de evaluación común para que las tres tareas sean comparables, y
partimos los datos **una sola vez** de forma estratificada, para que los tres modelos se
midan sobre exactamente los mismos pasajeros.

In [ ]:
X = titanic[['gender','class','edad','fare','embarked','titulo','edad_imputada']]
y = titanic['sobrevivio']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y)

print(f"Entrenamiento: {len(X_tr)}  |  Prueba: {len(X_te)}")
print(f"Tasa de supervivencia — train: {y_tr.mean():.3f}  test: {y_te.mean():.3f}")

resultados_titanic = []

def evaluar_clasificador(nombre, pipe, variables, guardar=True):
    # Entrena sobre un subconjunto de variables y devuelve las métricas
    pipe.fit(X_tr[variables], y_tr)
    pred = pipe.predict(X_te[variables])
    prob = pipe.predict_proba(X_te[variables])[:, 1]
    fila = {
        'algoritmo': nombre,
        'variables': ', '.join(variables),
        'exactitud': accuracy_score(y_te, pred),
        'precisión': precision_score(y_te, pred),
        'recall':    recall_score(y_te, pred),
        'F1':        f1_score(y_te, pred),
        'AUC':       roc_auc_score(y_te, prob),
    }
    if guardar:
        resultados_titanic.append(fila)
    return pipe, pred, prob, fila

### Algoritmo 1 · Supervivencia según el género

In [ ]:
prep_genero = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), ['gender'])
])

modelo_genero, pred_g, prob_g, fila_g = evaluar_clasificador(
    'A1 · Género',
    Pipeline([('prep', prep_genero),
              ('clf', LogisticRegression(random_state=SEMILLA))]),
    ['gender'])

print(classification_report(y_te, pred_g, target_names=['No sobrevivió','Sobrevivió'], digits=3))

# El coeficiente traducido a algo interpretable
coef = modelo_genero.named_steps['clf'].coef_[0][0]
print(f"Coeficiente de 'ser hombre': {coef:.3f}")
print(f"Razón de momios (odds ratio): {np.exp(coef):.3f}")
print(f"\nLectura: ser hombre multiplica las probabilidades relativas de sobrevivir")
print(f"por {np.exp(coef):.2f}, es decir, las divide por {1/np.exp(coef):.1f}.")

Un modelo con **una sola variable binaria** ya alcanza una exactitud alta. Esto no es
un logro del algoritmo: es la medida de cuánto determinó el género quién se salvó. El
modelo, en la práctica, aprendió la regla "si es mujer, predice que sobrevive".

### Algoritmo 2 · Supervivencia según la clase social

In [ ]:
prep_clase = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), ['class'])
])

modelo_clase, pred_c, prob_c, fila_c = evaluar_clasificador(
    'A2 · Clase social',
    Pipeline([('prep', prep_clase),
              ('clf', LogisticRegression(random_state=SEMILLA))]),
    ['class'])

print(classification_report(y_te, pred_c, target_names=['No sobrevivió','Sobrevivió'], digits=3))
print(f"Predicciones de 'sobrevivió': {pred_c.sum()} de {len(pred_c)}")

**Aquí hay un resultado que conviene leer con cuidado, no solo reportar.**

La exactitud parece razonable, pero el recall de la clase "Sobrevivió" es muy bajo. La
razón es que ninguna de las tres clases tiene tasa de supervivencia superior al 50%:
primera clase llega al 63% en el conjunto completo, pero al partir en train/test y con
solo esa variable el modelo termina prediciendo "no sobrevivió" para casi todos.

Es el ejemplo clásico de por qué **la exactitud sola engaña**: un modelo que predice
siempre la clase mayoritaria acierta el 62% de las veces sin haber aprendido nada. Por
eso reportamos también F1 y AUC, que sí distinguen ese caso.

### Algoritmo 3 · Supervivencia según la edad

In [ ]:
# Con la edad usamos un árbol y no una regresión logística, porque la relación
# no es monótona: los niños se salvan mucho, luego la tasa baja y se aplana.
prep_edad = ColumnTransformer([('num', SimpleImputer(strategy='median'), ['edad'])])

modelo_edad, pred_e, prob_e, fila_e = evaluar_clasificador(
    'A3 · Edad',
    Pipeline([('prep', prep_edad),
              ('clf', DecisionTreeClassifier(max_depth=3, min_samples_leaf=20,
                                             random_state=SEMILLA))]),
    ['edad'])

print(classification_report(y_te, pred_e, target_names=['No sobrevivió','Sobrevivió'], digits=3))

plt.figure(figsize=(14, 5))
plot_tree(modelo_edad.named_steps['clf'], feature_names=['edad'],
          class_names=['No sobrevivió','Sobrevivió'], filled=True, rounded=True, fontsize=10)
plt.title('Árbol de decisión sobre la edad')
plt.tight_layout(); plt.show()

In [ ]:
# Comparación honesta: ¿qué habría dado una regresión logística sobre la edad?
_, pred_e_log, prob_e_log, fila_e_log = evaluar_clasificador(
    'A3b · Edad (regresión logística)',
    Pipeline([('prep', ColumnTransformer([('num', StandardScaler(), ['edad'])])),
              ('clf', LogisticRegression(random_state=SEMILLA))]),
    ['edad'])

print(f"Árbol sobre edad     -> AUC = {fila_e['AUC']:.3f}")
print(f"Logística sobre edad -> AUC = {fila_e_log['AUC']:.3f}")
print("\nLa diferencia se debe a que la relación edad-supervivencia no es monótona:")
print("el árbol puede aislar el tramo infantil, la logística no.")

### Algoritmo 4 · Modelo completo (las tres variables combinadas)

No lo pide el enunciado de forma explícita, pero sin él no se puede responder la
pregunta que sí interesa: **cuánto aporta cada variable cuando compite con las demás.**

In [ ]:
VARS_NUM = ['edad','fare','edad_imputada']
VARS_CAT = ['gender','class','embarked','titulo']

prep_completo = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('esc', StandardScaler())]), VARS_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), VARS_CAT),
])

for nombre, clf in [
    ('A4 · Completo (logística)', LogisticRegression(max_iter=2000, random_state=SEMILLA)),
    ('A5 · Completo (árbol)',     DecisionTreeClassifier(max_depth=5, min_samples_leaf=10,
                                                        random_state=SEMILLA)),
    ('A6 · Completo (bosque)',    RandomForestClassifier(n_estimators=300, random_state=SEMILLA)),
]:
    _, pr, pb, fila = evaluar_clasificador(nombre, Pipeline([('prep', prep_completo),
                                                             ('clf', clf)]),
                                           VARS_NUM + VARS_CAT)
    print(f"{nombre:<28} exactitud={fila['exactitud']:.3f}  F1={fila['F1']:.3f}  AUC={fila['AUC']:.3f}")

In [ ]:
tabla_titanic = (pd.DataFrame(resultados_titanic)
                 .sort_values('AUC', ascending=False)
                 .reset_index(drop=True))
tabla_titanic.style.format({'exactitud':'{:.3f}','precisión':'{:.3f}','recall':'{:.3f}',
                            'F1':'{:.3f}','AUC':'{:.3f}'}).hide(axis='index')

In [ ]:
# Validación cruzada: ¿los resultados son estables o dependen de la partición?
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

print(f"{'Modelo':<30}{'AUC medio':>12}{'Desv.':>10}")
print('-'*52)
for nombre, variables, clf, prep in [
    ('Solo género', ['gender'], LogisticRegression(random_state=SEMILLA), prep_genero),
    ('Solo clase',  ['class'],  LogisticRegression(random_state=SEMILLA), prep_clase),
    ('Solo edad',   ['edad'],   DecisionTreeClassifier(max_depth=3, min_samples_leaf=20,
                                                      random_state=SEMILLA), prep_edad),
    ('Completo (bosque)', VARS_NUM+VARS_CAT,
     RandomForestClassifier(n_estimators=300, random_state=SEMILLA), prep_completo),
]:
    sc = cross_val_score(Pipeline([('prep', prep), ('clf', clf)]),
                         X[variables], y, cv=skf, scoring='roc_auc')
    print(f"{nombre:<30}{sc.mean():>12.4f}{sc.std():>10.4f}")

In [ ]:
# Curvas ROC: comparación visual de los tres algoritmos del enunciado
plt.figure(figsize=(7.5, 6.5))
for etiqueta, prob in [('Género', prob_g), ('Clase social', prob_c), ('Edad', prob_e)]:
    fpr, tpr, _ = roc_curve(y_te, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{etiqueta} (AUC = {roc_auc_score(y_te, prob):.3f})')

# El modelo completo, como referencia superior
mod_full = Pipeline([('prep', prep_completo),
                     ('clf', RandomForestClassifier(n_estimators=300, random_state=SEMILLA))])
mod_full.fit(X_tr[VARS_NUM+VARS_CAT], y_tr)
prob_full = mod_full.predict_proba(X_te[VARS_NUM+VARS_CAT])[:,1]
fpr, tpr, _ = roc_curve(y_te, prob_full)
plt.plot(fpr, tpr, lw=2.5, ls='--', color='black',
         label=f'Completo (AUC = {roc_auc_score(y_te, prob_full):.3f})')

plt.plot([0,1],[0,1],':',color='grey', label='Azar (AUC = 0.500)')
plt.xlabel('Falsos positivos'); plt.ylabel('Verdaderos positivos')
plt.title('Curvas ROC de los algoritmos del Titanic')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
# Importancia de variables en el modelo completo
nombres = mod_full.named_steps['prep'].get_feature_names_out()
imp = pd.DataFrame({
    'variable': [n.split('__')[-1] for n in nombres],
    'importancia': mod_full.named_steps['clf'].feature_importances_
}).sort_values('importancia', ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(data=imp.head(10), y='variable', x='importancia', palette='Blues_r')
plt.title('Qué pesó más al predecir la supervivencia'); plt.xlabel('Importancia')
plt.ylabel(''); plt.tight_layout(); plt.show()

print(imp.head(8).to_string(index=False))

In [ ]:
# Matrices de confusión de los tres algoritmos del enunciado
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a, (etiqueta, pred) in zip(ax, [('Género', pred_g), ('Clase social', pred_c), ('Edad', pred_e)]):
    cm = confusion_matrix(y_te, pred)
    ConfusionMatrixDisplay(cm, display_labels=['No sobr.','Sobrevivió']).plot(
        ax=a, cmap='Blues', colorbar=False, values_format='d')
    a.set_title(etiqueta); a.grid(False)
    a.set_xlabel('Predicho'); a.set_ylabel('Real' if a is ax[0] else '')
plt.tight_layout(); plt.show()

## 1.4 Conclusiones de la Parte 1

**El género es el factor decisivo.** Un modelo con esa sola variable alcanza casi el
mismo desempeño que el modelo completo. En términos de la época: la evacuación siguió
la norma "mujeres y niños primero" de forma bastante estricta.

**La clase social importa, pero mucho menos que el género.** Una mujer de tercera clase
tuvo mejor pronóstico que un hombre de primera. El dinero ayudaba, pero no compensaba.

**La edad solo discrimina en un tramo.** El efecto está concentrado en la niñez; entre
adultos la edad apenas cambia el pronóstico. Por eso el árbol supera a la regresión
logística en esa tarea: la relación no es monótona.

**Advertencia metodológica que conviene declarar.** El 20% de las edades fue imputado
por nosotros. Aunque usamos el título para hacerlo con criterio, sigue siendo un valor
estimado y no observado. Cualquier conclusión sobre la edad hereda esa incertidumbre.

---
---
# Parte 2 · Retrasos de vuelos (2015)

5.819.079 vuelos y 31 columnas, unos 592 MB en CSV. Aquí es donde el ejercicio se
vuelve realmente Big Data: cargarlo entero con pandas consume del orden de 2 a 3 GB de
RAM y en Google Colab puede tumbar el entorno.

**Dos técnicas para resolverlo, y las usamos ambas:**

1. **Leer solo las columnas necesarias** (`usecols`). De 31 columnas usamos 11. Eso
   solo ya reduce la memoria en más del 60%.
2. **Dask**, que divide el archivo en particiones y las procesa de a una.

## 2.1 Carga con Dask

In [ ]:
# Dask infiere el tipo de cada columna leyendo solo el inicio del archivo. Si en las
# primeras filas una columna se ve entera pero más adelante trae nulos, la inferencia
# falla con un error de tipos. Por eso los declaramos explícitamente.
TIPOS = {
    'MONTH':'int64', 'DAY':'int64', 'DAY_OF_WEEK':'int64',
    'AIRLINE':'object', 'ORIGIN_AIRPORT':'object',
    'SCHEDULED_DEPARTURE':'int64',
    'DEPARTURE_DELAY':'float64', 'ARRIVAL_DELAY':'float64',
    'DISTANCE':'int64', 'CANCELLED':'int64', 'DIVERTED':'int64',
}

t0 = time.time()
vuelos = dd.read_csv(RUTA_VUELOS, usecols=list(TIPOS), dtype=TIPOS, blocksize='64MB')

print(f"Particiones creadas: {vuelos.npartitions}")
print(f"Columnas leídas: {len(TIPOS)} de 31")
print(f"Tiempo de la lectura perezosa: {time.time()-t0:.2f} s")
print("\nOjo: este tiempo es casi cero porque Dask todavía no ha leído nada.")
print("Solo construyó el plan. Los datos se leen cuando se llama a .compute()")

In [ ]:
t0 = time.time()
n_vuelos = len(vuelos)
print(f"Total de vuelos: {n_vuelos:,}")
print(f"Ahora sí tardó: {time.time()-t0:.1f} s  (aquí ejecutó la lectura real)")

### Evaluación perezosa: el concepto central de Dask

Las dos celdas anteriores muestran de qué se trata. `dd.read_csv` devolvió en
milisegundos porque **no leyó el archivo**: solo anotó lo que habría que hacer. El
trabajo se disparó con `len()`, que obliga a materializar el resultado.

Eso se llama **evaluación perezosa** (*lazy evaluation*), y permite que Dask vea la
cadena completa de operaciones antes de ejecutar nada, para optimizarla. Es la
diferencia práctica más importante frente a pandas, donde cada línea se ejecuta de
inmediato.

In [ ]:
# Comparación de memoria: qué pasaría si cargáramos todo con pandas
muestra = pd.read_csv(RUTA_VUELOS, nrows=100_000)
mem_muestra = muestra.memory_usage(deep=True).sum() / 1e6
estimado = mem_muestra * (n_vuelos / 100_000) / 1000

muestra_10 = pd.read_csv(RUTA_VUELOS, nrows=100_000, usecols=list(TIPOS))
mem_10 = muestra_10.memory_usage(deep=True).sum() / 1e6
estimado_10 = mem_10 * (n_vuelos / 100_000) / 1000

print(f"Las 31 columnas con pandas   : ~{estimado:.1f} GB de RAM estimados")
print(f"Solo las 11 necesarias       : ~{estimado_10:.1f} GB estimados")
print(f"Ahorro solo por elegir columnas: {(1-estimado_10/estimado)*100:.0f}%")
print(f"\nGoogle Colab gratuito ofrece unos 12 GB. Con las 31 columnas y una copia")
print(f"intermedia, el margen se agota rápido.")

del muestra, muestra_10

## 2.2 Análisis exploratorio básico del conjunto de vuelos

In [ ]:
# Estructura general. Todo se calcula sobre las particiones de Dask.
t0 = time.time()
resumen_vuelos = pd.DataFrame({
    'nulos':   vuelos.isna().sum().compute(),
})
resumen_vuelos['% nulos'] = (resumen_vuelos['nulos']/n_vuelos*100).round(2)
resumen_vuelos['tipo'] = [str(t) for t in vuelos.dtypes]
print(f"Vuelos: {n_vuelos:,}   Columnas cargadas: {len(TIPOS)} de 31   ({time.time()-t0:.1f} s)\n")
print(resumen_vuelos[['tipo','nulos','% nulos']].to_string())

El 2,8% de nulos en `ARRIVAL_DELAY` no es un defecto del archivo: son los vuelos
**cancelados o desviados**, que nunca llegaron y por tanto no tienen retraso de llegada.
Es coherente con el 1,5% de cancelados que se ve enseguida.

In [ ]:
t0 = time.time()
estad = vuelos[['DEPARTURE_DELAY','ARRIVAL_DELAY','DISTANCE']].describe().compute()
cancel = vuelos['CANCELLED'].mean().compute()
desv   = vuelos['DIVERTED'].mean().compute()
print(f"({time.time()-t0:.1f} s)\n")
print(estad.round(2).to_string())
print(f"\nVuelos cancelados: {cancel:.2%}   |   desviados: {desv:.2%}")

In [ ]:
# ¿Cómo se reparten los retrasos? Categorías del Departamento de Transporte de EE.UU.
t0 = time.time()
llegados = vuelos[vuelos['ARRIVAL_DELAY'].notnull()]
categorias = {
    'Adelantado (< 0 min)':        (llegados['ARRIVAL_DELAY'] < 0).sum(),
    'A tiempo (0 a 15 min)':       ((llegados['ARRIVAL_DELAY'] >= 0) & (llegados['ARRIVAL_DELAY'] <= 15)).sum(),
    'Retraso leve (15 a 45 min)':  ((llegados['ARRIVAL_DELAY'] > 15) & (llegados['ARRIVAL_DELAY'] <= 45)).sum(),
    'Retraso alto (> 45 min)':     (llegados['ARRIVAL_DELAY'] > 45).sum(),
}
import dask
valores = dask.compute(*categorias.values())
reparto = pd.Series(valores, index=categorias.keys())
print(f"({time.time()-t0:.1f} s)\n")
print(pd.DataFrame({'vuelos': reparto, '%': (reparto/reparto.sum()*100).round(1)}).to_string())

In [ ]:
# Distribución de los retrasos. Se calcula el histograma de forma distribuida:
# solo los conteos por intervalo viajan a memoria, no los 5,8 millones de valores.
import dask.array as da

t0 = time.time()
vals = llegados['ARRIVAL_DELAY'].to_dask_array(lengths=True)
cnt, bordes_v = da.histogram(vals, bins=80, range=(-60, 180))
cnt = cnt.compute()
centros_v = (bordes_v[:-1] + bordes_v[1:]) / 2
print(f"Histograma distribuido en {time.time()-t0:.1f} s")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

colores = ['#55A868' if c <= 15 else '#C44E52' for c in centros_v]
ax[0].bar(centros_v, cnt/1000, width=(bordes_v[1]-bordes_v[0]), color=colores)
ax[0].axvline(0, color='black', lw=1)
ax[0].axvline(15, color='crimson', ls='--', label='umbral de 15 min')
ax[0].set(title='Distribución del retraso en la llegada',
          xlabel='Minutos (negativo = llegó antes)', ylabel='Miles de vuelos')
ax[0].legend()

ax[1].pie(reparto.values, labels=reparto.index, autopct='%1.1f%%',
          colors=['#55A868','#4C72B0','#DD8452','#C44E52'], startangle=90)
ax[1].set_title('Reparto de los vuelos por puntualidad')

plt.tight_layout(); plt.show()

La distribución está **centrada por debajo de cero**: la mayoría de los vuelos llega
antes de lo programado. Eso no es que las aerolíneas sean eficientes, es que los tiempos
de vuelo publicados incluyen un colchón. La cola derecha es larga pero delgada: pocos
vuelos con retrasos muy grandes arrastran el promedio hacia arriba, razón por la cual la
mediana y la media difieren tanto.

In [ ]:
# Aeropuertos con más tráfico y su puntualidad
t0 = time.time()
por_aeropuerto = vuelos.groupby('ORIGIN_AIRPORT').agg({
    'ARRIVAL_DELAY': ['count','mean']}).compute()
por_aeropuerto.columns = ['vuelos','retraso_prom']
top_aero = por_aeropuerto.sort_values('vuelos', ascending=False).head(12)
print(f"({time.time()-t0:.1f} s)\n")
print("Los 12 aeropuertos de origen con más tráfico:")
print(top_aero.round(2).to_string())

## 2.3 Las 10 aerolíneas con más retrasos

**Aquí hay una ambigüedad del enunciado que hay que resolver explícitamente.**
"Las 10 aerolíneas con más retrasos" puede significar dos cosas distintas, y dan
rankings **diferentes**:

- **Por cantidad**: cuántos vuelos retrasados acumula cada aerolínea. Favorece a las
  aerolíneas grandes, que simplemente vuelan más.
- **Por promedio**: cuántos minutos se retrasa en promedio cada vuelo. Mide la
  puntualidad real, sin importar el tamaño.

El enunciado pide después "agrupar por aerolínea y calcular el promedio de retraso", así
que el criterio principal es el promedio. Pero calculamos las dos y mostramos por qué
difieren, que es la parte interesante.

In [ ]:
# Nombres de las aerolíneas. El dataset trae solo el código IATA de dos letras,
# y el archivo airlines.csv de Kaggle no venía incluido, así que lo escribimos.
AEROLINEAS = {
    'UA':'United Air Lines',      'AA':'American Airlines',   'US':'US Airways',
    'F9':'Frontier Airlines',     'B6':'JetBlue Airways',     'OO':'Skywest Airlines',
    'AS':'Alaska Airlines',       'NK':'Spirit Air Lines',    'WN':'Southwest Airlines',
    'DL':'Delta Air Lines',       'EV':'Atlantic Southeast',  'HA':'Hawaiian Airlines',
    'MQ':'American Eagle',        'VX':'Virgin America',
}

t0 = time.time()
agg = vuelos.groupby('AIRLINE').agg({
    'ARRIVAL_DELAY': ['mean','count'],
    'DEPARTURE_DELAY': 'mean',
    'CANCELLED': 'sum',
}).compute()
agg.columns = ['retraso_llegada_prom','vuelos','retraso_salida_prom','cancelados']
agg['aerolinea'] = agg.index.map(AEROLINEAS)
print(f"Agregación de {n_vuelos:,} filas en {time.time()-t0:.1f} s")

In [ ]:
# Criterio A: promedio de minutos de retraso por vuelo
top_promedio = agg.sort_values('retraso_llegada_prom', ascending=False).head(10)
print("TOP 10 POR RETRASO PROMEDIO (minutos de retraso en la llegada)")
print(top_promedio[['aerolinea','retraso_llegada_prom','vuelos']]
      .round(2).to_string(index=True))

In [ ]:
# Criterio B: cantidad de vuelos retrasados (más de 15 minutos, criterio del
# Departamento de Transporte de EE.UU. para considerar un vuelo "retrasado")
t0 = time.time()
retrasados = vuelos[vuelos['ARRIVAL_DELAY'] > 15]
conteo = retrasados.groupby('AIRLINE').size().compute().sort_values(ascending=False)
print(f"({time.time()-t0:.1f} s)\n")

tabla_conteo = pd.DataFrame({
    'aerolinea': conteo.index.map(AEROLINEAS),
    'vuelos_retrasados': conteo.values,
    'vuelos_totales': agg.loc[conteo.index, 'vuelos'].values,
})
tabla_conteo['% retrasados'] = (tabla_conteo['vuelos_retrasados'] /
                                tabla_conteo['vuelos_totales'] * 100).round(1)
print("TOP 10 POR CANTIDAD DE VUELOS RETRASADOS (más de 15 min)")
print(tabla_conteo.head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))

t = top_promedio.iloc[::-1]
colores = ['#C44E52' if v > 10 else '#4C72B0' for v in t['retraso_llegada_prom']]
ax[0].barh(t['aerolinea'], t['retraso_llegada_prom'], color=colores)
ax[0].set_title('Top 10 por retraso PROMEDIO')
ax[0].set_xlabel('Minutos de retraso promedio en la llegada')
for i, v in enumerate(t['retraso_llegada_prom']):
    ax[0].text(v + 0.2, i, f'{v:.1f}', va='center', fontsize=9)

t2 = tabla_conteo.head(10).iloc[::-1]
ax[1].barh(t2['aerolinea'], t2['vuelos_retrasados']/1000, color='#DD8452')
ax[1].set_title('Top 10 por CANTIDAD de vuelos retrasados')
ax[1].set_xlabel('Miles de vuelos con más de 15 min de retraso')

plt.tight_layout(); plt.show()

### Por qué los dos rankings no coinciden

**Southwest** encabeza el ranking por cantidad y queda a media tabla por promedio.
No es que sea especialmente impuntual: opera muchísimos más vuelos que las demás, así
que acumula más retrasos en términos absolutos aunque cada vuelo suyo se retrase poco.

**Spirit y Frontier** son lo contrario: vuelan relativamente poco, pero cuando vuelan
se retrasan mucho. Son las peores en puntualidad real.

Para un pasajero que elige aerolínea, el ranking útil es el del **promedio**. Para un
regulador que quiere reducir el total de horas perdidas por los viajeros, el útil es el
de **cantidad**. La misma pregunta admite dos respuestas correctas según a quién se le
responda.

## 2.4 Variación de los retrasos a lo largo del año

In [ ]:
t0 = time.time()
por_mes = vuelos.groupby('MONTH').agg({
    'ARRIVAL_DELAY': ['mean','count'],
    'DEPARTURE_DELAY': 'mean',
    'CANCELLED': 'mean',
}).compute()
por_mes.columns = ['retraso_prom','vuelos','retraso_salida','tasa_cancelacion']
por_mes = por_mes.sort_index()
por_mes['tasa_cancelacion'] = por_mes['tasa_cancelacion']*100

MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
por_mes['mes'] = MESES
print(f"({time.time()-t0:.1f} s)")
print(por_mes[['mes','retraso_prom','vuelos','tasa_cancelacion']].round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

colores = ['#C44E52' if v > por_mes['retraso_prom'].mean() else '#4C72B0'
           for v in por_mes['retraso_prom']]
ax[0].bar(por_mes['mes'], por_mes['retraso_prom'], color=colores)
ax[0].axhline(por_mes['retraso_prom'].mean(), color='grey', ls='--',
              label=f"promedio anual = {por_mes['retraso_prom'].mean():.1f} min")
ax[0].set_title('Retraso promedio de llegada por mes')
ax[0].set_ylabel('Minutos'); ax[0].legend()

ax[1].plot(por_mes['mes'], por_mes['tasa_cancelacion'], marker='o', lw=2.5, color='#C44E52')
ax[1].set_title('Tasa de cancelación por mes')
ax[1].set_ylabel('% de vuelos cancelados')

plt.tight_layout(); plt.show()

In [ ]:
# ¿Y a lo largo del día y de la semana?
t0 = time.time()
vuelos_h = vuelos.assign(HORA=(vuelos['SCHEDULED_DEPARTURE'] // 100))
por_hora = vuelos_h.groupby('HORA')['ARRIVAL_DELAY'].mean().compute().sort_index()
por_dia  = vuelos.groupby('DAY_OF_WEEK')['ARRIVAL_DELAY'].mean().compute().sort_index()
print(f"({time.time()-t0:.1f} s)")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(por_hora.index, por_hora.values, marker='o', color='#4C72B0')
ax[0].axhline(0, color='grey', lw=.8)
ax[0].set(title='Retraso promedio según la hora de salida programada',
          xlabel='Hora del día', ylabel='Minutos de retraso')
ax[0].set_xticks(range(0,24,2))

DIAS = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
ax[1].bar(DIAS, por_dia.values, color='#DD8452')
ax[1].set(title='Retraso promedio según el día de la semana', ylabel='Minutos')

plt.tight_layout(); plt.show()

### Lectura de la Parte 2

**Junio es el peor mes** (9,6 minutos de retraso promedio), seguido de febrero, julio y
diciembre. Junio y julio coinciden con las vacaciones de verano y diciembre con las
fiestas: más pasajeros, más vuelos y menos margen para absorber cualquier incidente.
Febrero es un caso aparte y lo explicamos enseguida.

**Septiembre y octubre tienen retraso promedio negativo**, es decir que en promedio los
vuelos llegaron antes de lo programado. Es la temporada baja entre el verano y las
fiestas.

**Febrero tiene la tasa de cancelación más alta** aunque no el mayor retraso promedio.
Tiene sentido: las tormentas de invierno cancelan vuelos en lugar de retrasarlos, y un
vuelo cancelado no aporta minutos de retraso porque directamente desaparece de esa
estadística. Es un sesgo que conviene mencionar: **el retraso promedio subestima el mal
desempeño de los meses de invierno.**

**Los vuelos de la mañana llegan antes de tiempo.** El retraso promedio es negativo
hasta media mañana y crece durante todo el día hasta la noche. La razón es el efecto
acumulativo: un avión que sale tarde a las 7 a.m. arrastra ese retraso en todos sus
vuelos siguientes. Recomendación práctica que sale de los datos: **si puede, vuele
temprano.**

---
---
# Parte 3 · Viajes de taxi en Nueva York

1.458.644 viajes de 2016. El enunciado pide tres cosas: investigar qué hace Dask,
analizar la distribución de las tarifas, y ver cómo varían los viajes a lo largo del
día.

## 3.1 ¿Qué hace la librería Dask?

**Dask es una librería de Python para computación paralela que permite trabajar con
conjuntos de datos más grandes que la memoria RAM disponible.**

### El problema que resuelve

Pandas carga todo el archivo en memoria de una vez. Si el archivo pesa 10 GB y el
computador tiene 8 GB de RAM, el programa se cae. Dask evita eso dividiendo el trabajo.

### Cómo funciona

Un DataFrame de Dask **no es** una tabla: es un conjunto de DataFrames de pandas
(llamados **particiones**) más un plan de qué hacer con ellos. Cuando se pide una
operación, Dask la aplica partición por partición y combina los resultados. En cualquier
momento solo hay unas pocas particiones en memoria.

| Concepto | Qué significa |
|---|---|
| **Partición** | Un trozo del archivo, que sí es un DataFrame de pandas normal |
| **Evaluación perezosa** | Nada se ejecuta hasta que se llama a `.compute()` |
| **Grafo de tareas** | El plan de operaciones que Dask construye y optimiza antes de ejecutar |
| **Paralelismo** | Si hay varios núcleos, procesa varias particiones a la vez |

### Diferencias prácticas con pandas

| | pandas | Dask |
|---|---|---|
| Ejecución | Inmediata | Perezosa, hasta `.compute()` |
| Memoria | Todo el archivo | Solo unas particiones a la vez |
| Tamaño máximo | Limitado por la RAM | Limitado por el disco |
| Velocidad en datos pequeños | Más rápido | Más lento (el reparto cuesta) |
| API | — | Casi idéntica a la de pandas |

### Cuándo NO usar Dask

Esto es lo que suele faltar en las respuestas de este ejercicio. **Dask es más lento
que pandas cuando los datos caben en memoria**, porque dividir, coordinar y volver a
juntar tiene un costo fijo. La regla práctica: por debajo de 1 GB, pandas; por encima
de la RAM disponible, Dask.

Por eso en la Parte 1 de este trabajo usamos pandas: con 891 filas, Dask sería una mala
decisión técnica. Lo demostramos con números más abajo.

### Alternativas

Dask no es la única opción. **Polars** suele ser más rápido en una sola máquina,
**PySpark** es el estándar en clústeres grandes, y el propio pandas admite lectura por
trozos con `chunksize`. Dask tiene la ventaja de que su API es casi la misma de pandas,
así que el código existente casi no cambia.

## 3.2 Carga de los datos de taxis

In [ ]:
TIPOS_TAXI = {
    'id':'object', 'vendor_id':'int64', 'passenger_count':'int64',
    'pickup_longitude':'float64', 'pickup_latitude':'float64',
    'dropoff_longitude':'float64','dropoff_latitude':'float64',
    'store_and_fwd_flag':'object','trip_duration':'int64',
}

taxis = dd.read_csv(RUTA_TAXIS, dtype=TIPOS_TAXI,
                    parse_dates=['pickup_datetime','dropoff_datetime'],
                    blocksize='32MB')

print(f"Particiones: {taxis.npartitions}")
print(f"Columnas: {list(taxis.columns)}")
n_taxis = len(taxis)
print(f"\nViajes: {n_taxis:,}")

**Confirmación de lo que anticipamos al inicio:** las columnas disponibles son
identificador, proveedor, fechas de recogida y entrega, número de pasajeros,
coordenadas, una bandera técnica y la duración del viaje. **No hay ninguna columna de
tarifa.** El enunciado pide analizar tarifas sobre un conjunto que no las contiene.

In [ ]:
# Demostración empírica de cuándo Dask conviene y cuándo no
print("¿Cuánto tarda una misma operación en pandas y en Dask?\n")

# (a) Datos pequeños: el Titanic, 891 filas
t0 = time.time(); titanic.groupby('class')['sobrevivio'].mean(); t_pd_small = time.time()-t0
tit_dask = dd.from_pandas(titanic, npartitions=4)
t0 = time.time(); tit_dask.groupby('class')['sobrevivio'].mean().compute(); t_dk_small = time.time()-t0

print(f"891 filas  -> pandas: {t_pd_small*1000:7.1f} ms | Dask: {t_dk_small*1000:7.1f} ms"
      f"  ({t_dk_small/t_pd_small:.0f}x MÁS LENTO con Dask)")

# (b) Datos grandes: los taxis, 1.45 millones de filas
t0 = time.time()
_ = taxis.groupby('vendor_id')['trip_duration'].mean().compute()
t_dk_big = time.time()-t0
t0 = time.time()
tmp = pd.read_csv(RUTA_TAXIS, usecols=['vendor_id','trip_duration'])
_ = tmp.groupby('vendor_id')['trip_duration'].mean()
t_pd_big = time.time()-t0
del tmp

print(f"1.4M filas -> pandas: {t_pd_big:7.1f} s  | Dask: {t_dk_big:7.1f} s")
print("\nConclusión: la ventaja de Dask no es la velocidad pura, es poder procesar")
print("archivos que no caben en memoria sin que el programa se caiga.")

## 3.3 Análisis exploratorio básico del conjunto de taxis

In [ ]:
print("Primeras filas:")
taxis.head()

In [ ]:
t0 = time.time()
nulos_taxi = taxis.isna().sum().compute()
print(f"Nulos por columna ({time.time()-t0:.1f} s):")
print(nulos_taxi.to_string())
print(f"\nRango de fechas: {taxis['pickup_datetime'].min().compute()}"
      f"  a  {taxis['pickup_datetime'].max().compute()}")

In [ ]:
t0 = time.time()
pasajeros = taxis['passenger_count'].value_counts().compute().sort_index()
proveedor = taxis['vendor_id'].value_counts().compute().sort_index()
print(f"({time.time()-t0:.1f} s)\n")
print("Pasajeros por viaje:")
print(pd.DataFrame({'viajes': pasajeros,
                    '%': (pasajeros/n_taxis*100).round(2)}).to_string())
print("\nProveedor:")
print(proveedor.to_string())

Siete de cada diez viajes (70,9%) llevan **un solo pasajero**. También aparecen viajes con
cero pasajeros, que son un error de registro del taxímetro: un viaje sin nadie a bordo no
es un viaje. Son pocos y no afectan el análisis, pero conviene notarlos.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

p = pasajeros[pasajeros.index.isin(range(0,7))]
ax[0].bar(p.index.astype(str), p.values/1000, color='#4C72B0')
ax[0].set(title='Viajes según número de pasajeros',
          xlabel='Pasajeros', ylabel='Miles de viajes')

por_mes_taxi = taxis.groupby(taxis['pickup_datetime'].dt.month).size().compute().sort_index()
MESES_T = ['Ene','Feb','Mar','Abr','May','Jun']
ax[1].bar(MESES_T[:len(por_mes_taxi)], por_mes_taxi.values/1000, color='#DD8452')
ax[1].set(title='Viajes por mes', ylabel='Miles de viajes')

plt.tight_layout(); plt.show()
print("El conjunto cubre solo el primer semestre de 2016.")

## 3.4 Limpieza

In [ ]:
# La duración viene en segundos. Hay valores absurdos conocidos en este conjunto:
# viajes de 1 segundo y viajes de más de 24 horas (errores del taxímetro).
desc = taxis['trip_duration'].describe().compute()
print("Duración de los viajes, en segundos:")
print(desc.round(1).to_string())
print(f"\nEl máximo son {desc['max']/3600:,.0f} horas. Claramente es un error de registro.")

In [ ]:
# Filtramos a viajes plausibles: entre 1 minuto y 3 horas.
# Un viaje en taxi de menos de un minuto no es un viaje; uno de más de tres horas
# dentro de Nueva York tampoco es un trayecto urbano normal.
antes = n_taxis
taxis_limpio = taxis[(taxis['trip_duration'] >= 60) & (taxis['trip_duration'] <= 10800)]
n_limpio = len(taxis_limpio)

print(f"Viajes originales : {antes:,}")
print(f"Viajes conservados: {n_limpio:,}")
print(f"Eliminados        : {antes-n_limpio:,} ({(antes-n_limpio)/antes:.2%})")

## 3.5 Distribución de las tarifas

Como el conjunto no trae tarifas, procedemos en dos pasos:

1. **La duración del viaje**, que es el dato real y medido.
2. **Una tarifa estimada**, aplicando la fórmula oficial de la Taxi & Limousine
   Commission de Nueva York vigente en 2016: 2,50 USD de bajada de bandera más
   0,50 USD por cada minuto detenido o a baja velocidad. Como no tenemos el detalle
   por segundo, aplicamos la tarifa por tiempo sobre la duración total.

**La estimación es nuestra, no un dato del conjunto.** Sirve para responder la pregunta
en las unidades que pide el enunciado, pero cualquier conclusión sobre montos hay que
leerla como aproximación. Lo tomamos como una cota inferior, porque la tarifa real
incluye además un componente por distancia recorrida.

In [ ]:
BAJADA_BANDERA = 2.50     # USD, tarifa inicial (TLC NYC, 2016)
POR_MINUTO     = 0.50     # USD por minuto

taxis_limpio = taxis_limpio.assign(
    duracion_min = taxis_limpio['trip_duration'] / 60,
    tarifa_estimada = BAJADA_BANDERA + (taxis_limpio['trip_duration']/60) * POR_MINUTO,
)

t0 = time.time()
est_dur = taxis_limpio['duracion_min'].describe().compute()
est_tar = taxis_limpio['tarifa_estimada'].describe().compute()
print(f"({time.time()-t0:.1f} s)\n")

comparacion = pd.DataFrame({'duración (min)': est_dur, 'tarifa estimada (USD)': est_tar})
print(comparacion.round(2).to_string())

In [ ]:
# Histograma calculado de forma distribuida: Dask solo trae a memoria los conteos
# de cada intervalo, nunca los 1.4 millones de valores.
import dask.array as da

t0 = time.time()
valores = taxis_limpio['duracion_min'].to_dask_array(lengths=True)
conteos, bordes = da.histogram(valores, bins=60, range=(1, 90))
conteos = conteos.compute()
centros = (bordes[:-1] + bordes[1:]) / 2
print(f"Histograma distribuido calculado en {time.time()-t0:.1f} s")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

ax[0].bar(centros, conteos, width=(bordes[1]-bordes[0]), color='#4C72B0')
ax[0].axvline(est_dur['50%'], color='crimson', ls='--',
              label=f"mediana = {est_dur['50%']:.1f} min")
ax[0].set(title='Distribución de la duración de los viajes',
          xlabel='Minutos', ylabel='Nº de viajes'); ax[0].legend()

tarifas_centros = BAJADA_BANDERA + centros * POR_MINUTO
ax[1].bar(tarifas_centros, conteos, width=(bordes[1]-bordes[0])*POR_MINUTO, color='#DD8452')
ax[1].axvline(est_tar['50%'], color='crimson', ls='--',
              label=f"mediana = ${est_tar['50%']:.2f}")
ax[1].set(title='Distribución de la tarifa estimada',
          xlabel='USD (estimado, no medido)', ylabel=''); ax[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Percentiles: más informativos que la media en una distribución sesgada
t0 = time.time()
pcts = taxis_limpio['duracion_min'].quantile([.1,.25,.5,.75,.9,.95,.99]).compute()
print(f"({time.time()-t0:.1f} s)\n")

tabla_p = pd.DataFrame({
    'percentil': ['10%','25%','50% (mediana)','75%','90%','95%','99%'],
    'duración (min)': pcts.values.round(1),
    'tarifa estimada (USD)': (BAJADA_BANDERA + pcts.values*POR_MINUTO).round(2),
})
print(tabla_p.to_string(index=False))
print(f"\nLa distribución es asimétrica: la media ({est_dur['mean']:.1f} min) es mayor que")
print(f"la mediana ({est_dur['50%']:.1f} min), señal de una cola de viajes largos.")

## 3.6 Variación de los viajes a lo largo del día

In [ ]:
t0 = time.time()
taxis_hora = taxis_limpio.assign(
    hora = taxis_limpio['pickup_datetime'].dt.hour,
    dia_semana = taxis_limpio['pickup_datetime'].dt.dayofweek,
)

por_hora_taxi = taxis_hora.groupby('hora').agg({
    'trip_duration': ['count','mean'],
    'passenger_count': 'mean',
}).compute()
por_hora_taxi.columns = ['viajes','duracion_prom_seg','pasajeros_prom']
por_hora_taxi = por_hora_taxi.sort_index()
por_hora_taxi['duracion_prom_min'] = por_hora_taxi['duracion_prom_seg']/60
por_hora_taxi['tarifa_estimada'] = BAJADA_BANDERA + por_hora_taxi['duracion_prom_min']*POR_MINUTO

print(f"({time.time()-t0:.1f} s)")
print(por_hora_taxi[['viajes','duracion_prom_min','tarifa_estimada','pasajeros_prom']]
      .round(2).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

pico = por_hora_taxi['viajes'].idxmax()
valle = por_hora_taxi['viajes'].idxmin()
colores = ['#C44E52' if h == pico else '#55A868' if h == valle else '#4C72B0'
           for h in por_hora_taxi.index]
ax[0].bar(por_hora_taxi.index, por_hora_taxi['viajes']/1000, color=colores)
ax[0].set(title='Cantidad de viajes por hora del día',
          xlabel='Hora', ylabel='Miles de viajes')
ax[0].set_xticks(range(0,24,2))
ax[0].annotate(f'pico: {pico}:00', xy=(pico, por_hora_taxi['viajes'].max()/1000),
               xytext=(pico-5, por_hora_taxi['viajes'].max()/1000*0.9),
               arrowprops=dict(arrowstyle='->', color='crimson'), color='crimson')

ax[1].plot(por_hora_taxi.index, por_hora_taxi['duracion_prom_min'],
           marker='o', color='#DD8452', lw=2.5)
ax[1].set(title='Duración promedio del viaje por hora',
          xlabel='Hora', ylabel='Minutos')
ax[1].set_xticks(range(0,24,2))

plt.tight_layout(); plt.show()

In [ ]:
# Cruce hora x día de la semana: el patrón entre semana y el de fin de semana
t0 = time.time()
mapa = taxis_hora.groupby(['dia_semana','hora'])['trip_duration'].count().compute()
print(f"({time.time()-t0:.1f} s)")

matriz = mapa.unstack(fill_value=0).sort_index()
matriz.index = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']

plt.figure(figsize=(14, 4.5))
sns.heatmap(matriz/1000, cmap='YlOrRd', cbar_kws={'label':'miles de viajes'},
            linewidths=.3)
plt.title('Viajes por hora y día de la semana')
plt.xlabel('Hora del día'); plt.ylabel('')
plt.tight_layout(); plt.show()

### Lectura de la Parte 3

**El pico es al final de la tarde**, entre las 18:00 y las 19:00, con la salida de las
oficinas. El valle está de madrugada, alrededor de las 5:00, la única hora en que Nueva
York casi se detiene.

**Los viajes más lentos no coinciden con los más numerosos.** La duración promedio
alcanza su máximo entre las 15:00 y las 16:00, cuando el tráfico está peor, no a las
18:00 que es cuando hay más viajes. La hora de mayor demanda no es la hora en que peor
se mueve la ciudad.

**El mapa de calor muestra dos patrones distintos.** Entre semana hay dos franjas
marcadas, mañana y tarde, típicas de los desplazamientos al trabajo. El fin de semana
la actividad se corre hacia la noche y se extiende hasta la madrugada del sábado y el
domingo, que es ocio y no trabajo.

**Implicación de tarifas:** como la tarifa por tiempo crece con la duración, viajar a
mediodía sale más caro por el mismo trayecto que viajar de madrugada. El pico de demanda
y el pico de costo no ocurren a la misma hora.

---
---
# Conclusiones generales

### Parte 1 · Titanic
El género fue el factor determinante de la supervivencia: un modelo con esa única
variable alcanza casi el desempeño del modelo completo. La clase social influyó de forma
clara pero secundaria, y la edad solo discriminó en el tramo infantil, razón por la cual
un árbol supera a la regresión logística en esa tarea. Declaramos que el 20% de las
edades fue imputado por nosotros usando el título del pasajero.

### Parte 2 · Retrasos aéreos
Los dos rankings de "aerolíneas con más retrasos" no coinciden y ambos son válidos según
a quién se responda: por volumen encabezan las aerolíneas grandes, por puntualidad real
encabezan las de bajo costo. Los retrasos crecen en verano y a fin de año, y aumentan a
lo largo del día por acumulación. El retraso promedio subestima el mal desempeño
invernal, porque en invierno los vuelos se cancelan en lugar de retrasarse.

### Parte 3 · Taxis y Dask
Dask permite procesar archivos mayores que la memoria disponible dividiéndolos en
particiones y evaluando de forma perezosa, con una API casi idéntica a la de pandas.
Su ventaja no es la velocidad: en datos pequeños es varias veces más lento, como
medimos. La demanda de taxis alcanza su pico al final de la tarde, pero la duración de
los viajes es máxima cerca del mediodía, así que el pico de demanda y el de costo no
coinciden.

### Limitaciones que declaramos

1. **El conjunto de taxis no contiene tarifas.** Las tarifas de este trabajo son una
   estimación nuestra a partir de la duración y la fórmula de la TLC de 2016. No son un
   dato medido, y al no incluir el componente por distancia son una cota inferior.
2. **El 20% de las edades del Titanic fue imputado.** Se hizo con criterio, usando el
   título del pasajero, pero las conclusiones sobre edad heredan esa incertidumbre.
3. **El archivo del Titanic no es el estándar de Kaggle** y carece de las variables
   familiares (`SibSp`, `Parch`), que en la literatura resultan predictivas.
4. **Los datos de vuelos son solo de 2015**, un año concreto de un solo país. Las
   conclusiones estacionales no son necesariamente extrapolables.